<a href="https://colab.research.google.com/github/progstay17/mCAP-to-mp4-drive-converter/blob/main/mcap_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCAP → MP4 (folder Drive, satu per satu)
Jalankan sel **berurutan**. Berhenti dan cek hasil di tiap langkah sebelum lanjut.

Cara kerja: tiap file `.mcap` disalin ke disk lokal Colab → diproses di proses Python terpisah (RAM bebas total setelah selesai) → MP4 disalin ke Drive → file lokal dihapus.

## Langkah 1 — Hubungkan Drive & siapkan skrip
Skrip ekstraksi sudah tertanam di notebook ini, tidak perlu upload apa pun. Mendukung dua jenis file: `ego_*.mcap` (dua kamera, protobuf) dan file ROS2 bertopik `/head/camera/image_compressed` (satu kamera, H.265). Jenisnya dikenali otomatis. Chunk `zstd` dan `lz4` juga didukung.

In [ ]:
import base64, zlib, hashlib, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# dukungan chunk MCAP terkompresi (zstd / lz4)
for pkg in ('zstandard', 'lz4'):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'PERINGATAN: gagal install {pkg}:', r.stderr.strip()[-200:])

_FILES = {
    'mcap_to_mp4.py': ('eNrNPGtzHFV23/Ureju10I3GoxnNWIiJh4Q4uKAqpJwy4UPkqa5Wzx2p1z3dne6ekWShKi/FwyaAnWUXdrG3KLJgttjw2IQCY7PwXzYeSf7EX8g55z769mNGXiofIlbW3Ne555x73vfOjpJobDjOaJJNEuY4hj+OoyQz3DCMMjfzozBdWpJ9yVbsJimT7Z+lUSg/p1ky8TLZyvwxWxoh5KGbuV7gpilLJWjV1TBGPguGfGLsZtuBvyknnYfm0tLS2aeeu2D0jX1zhW1FK547Zom70jJ7hsk/Oy2zYRQG29pg2zwAGEM2Mjbba9bU7hkJAzJDgW0zdr1LlvnkMwBjavN5q1193rSZRc7mXsZSqwP7bPpbppjXWZ0P71kNXrRrZQ0jzqfiyoCFVmwby8Y6/pPBb0yzR2L6FDgTFFfxAY7KxnSA6xDXUYCfYr7ZBIjcbBhRBa9JiJg5yGfLPIPk0rSN1gD+wPpVvhwwe4jlz1aWd/nyte7DLP+nyvJ1cUZjmC8gLBnwE+JwP0eLOiU/NqIerg0HzSHzoiGzzEk2OrWO0pCwOHA9ZtocfCjBc+bVbCARr9+gCCW95MfO2I0fFlF97dRN/DDTV05hWYs+perTzrYfMOP5ZML4HPzxYBQwGqgOANs32qo5NV7sG5ZnPGK0dh8/ZxtnzhipGvRHsP4MjKy3cogaiiBtkepPEfDjAmEvGscJA90dOlN/yCIndveCyB1a43RLUBApvFGtofGPUcioPRpnepOTFQEiKP05APy5xPY4CwWHYFTxkUChmYBhmGc8+aTRUQM7meh9BHDW6MV+wKtIrrNwD2JzFGZ+OGFVUO0iqKkbACQAIaRkfVAYptNZV10sUHBWi3DCE1Eq7xTW7BTW7XR6McbdGjhdDU7KSrLi+ikzXnCDCXs6SaLEGpmTMJ3EaKzZ0IiTKIs2J7C5nzAj24uZsb+THZh2lZWr4FuG8kj7Rqe4j5AiQLeWfcW13eJaLnGwdIFNaAbRDkusgo7ing1cLcTez1jiJAAiGaYW+qUG0reFmtBHeRaSm4GDDLh3gG1xXjMFl2nZ8MdJ/ctcjAB3ubinPlmthr6cY7PjZ9scTBSDipjJpmkbbmqMejoTR82EgQaC5/gJ2ATz4u76E8+dfep862JyMTRPPLN9hH9g+KkB7h2YaeBaYGjAtKOqs0DzSRk1MxYEll1DkZKuGBgkEG/bZZiISRQXt8pPp0TRjgPmI4e2XoGG1kVMIxatPxxg5uVAC35LuCwJErxWZcsohl7cq7XbWqtuV7Eq0m6uo+UQv92SIRrH3DSQUwTsKobBD0OWiDncs9XO4kYclcW8nGZDs4qeCLhgFHTLTYbVCbgTbKNmNP8FIP09k94BBMsmjeNNqwJAQ3fs7jrRJIsnXEP6c0MEIqa9VuU3WQNFU3C5O58kGGyOEggF55KkZuj40+CcTWuEabFhzF2o4W1Pwks9Yx+7dMtICOV+tKiFvvSYHKsaWkm5aHTDjwY148VQQfVzPZp7AOLAfArU5kBdr/TvkWkGnBoKpR7Ope3mgcGx6tBCY2Oc0piC4KOS+RHRWhz4mQO5DNt1Ni2084KDIMpJhmZ7gyPll8IvX3CdlsBmnYINxt4Nv+eTH0U5BDvcov/aJWnk+zTdGCz60LL8htG1Sydf9rxF+B0d/snQO3XQOwv8up9LB+hlzpBRlBj+cLdhWLQJeEAyqH5osHCCyVXG+EiqyWSYkkRBLyYl+qmGTI6kGwAWhtsDNJpArGgKhguYhGl+Ajkc0DOARAwK017IBs0ERNiHeBxYVAw3cG6RWKBQMgsHi4H6RLr/EMMXPkGlMtREhCH8bNsUZ3fOifnJZhqL6Vy2EvQltGC1N9BYi6LpJom7J8KPyyyJ8rgfOe4hg2F9Qdr4tCdl9OPVRE1FSHMdj0a/l7NKLuZ/8SgoY8AAmh9Dq5AYka8BSKhjlMYbf+dnKccH2eE4fghK51gpC0YNQ1M6Yg50NofiCIu9XAcVHO6KCUaoAZgWyESmOci0xA234NBKBhInW1NMh+DQXjQsi2+/IfZbWYHIHY/Uehx0XHT+FLJyPOFSoCJGC7ZUJlA50pt+RntoSTCuk5GPmjdhYtrS/CPklkhfX5PUiEOrwcpqI900jhasDSebw+LdcIJ/q6ox4jCfDUdRT8Sjw2y7B8xF2W0/scox22b+1namulvrvBssNEaRTgp+hMlRPpT5LCn2yMn+0FOAlmTc4yrgrd21Fv8RgyGougtj4BFk2K0sL/1n0sSATVmgAAu8vW3wbW5xu2Aydp0hi3My9bl1IxnD0ALcTODusSQtglODIUvB9+eD3E5IqpEfDuFobYqQKJ1sClEo8BHpS/iBrdqKl1qvkFGNn9rgaVvjqdbfWbXL/NRGu+t2XvRaU0UvxVdtqgi+4zh3G0FcdCGabhbp5AulNQKIqDma8wrmjIFpEoB684zAOkh7dTP8KTGTfNwiBDEGiTf8wRwoZCuXjdPoprHRXS/4n2D+0vWC6ymceYPOuKGfaUOcYUM/sgY/DilbWJcFAKkFv4KCTZQUtM0WeSgcsJd0LLq8KejWJYQPcDGuEbdUIVnBrqGk5ARxl5iAJbQ1peO7qU70RKK/4PRKGJGpKi3lhqoKT8pTDixgI8A+wekYRULsuhllWTTO1zbmf8h92A7quoYxCBbm+R0RzKzmMxHV1SJtbT4pt+KcpFN9hPuYYSGOIGKEZL6noBFnbeMsQB4Lw4S90FkyYiUueHWdQhjR/lu0eUOAbzzMgTcELQ2xY0Nsog4claUhRKrW8VxwxxAy9lQtUdj4JVEghGYUBXULn08gdeHrIEvvGYGfZhvZBIBt0EcCM2jQ+sEA03xMUixQG3cSZM7I9bIo2evjXCHfLqSEYNuHvpdtgK41DA3OAgDueHMIaO93VntgA+HkO+JvF/8eCAlE/0pcngsIB4WF8iG65X7kRSqlyooq1/pJ6P+r5eJlQ09K95TKKSB6bi7frjSlU5kXucOhquVmyD44Wd6kOhhpge/lULGyRpU3+PsTSLu3V9dOa7lIXaGJIFClaR9WHTQIL1onjAvMpnShkKIJLISzgzgcp+yLiNwmix0idbT6QFnxvC9HClVfLtTNMnGHiG6Kgy6YaOKpPryRDTACLZdVqB7Z6RBbME8pLOl0BnZN2ZjQonk+P//cbIsNsJbdB5h7VnvNONOHXeCf1TYRyRFHnihbxoGRlCCnSUIQIQSDf13IlUu05KAgEEAZRfm0NWuow+xrpIFuiWORTZV4EucbuKttq6uDIGAeHCxgML+E6m3jwQd0yAc5cGqTAPW4dlv86KkLUcdrwfz00WJjAQ8GFpVu5Yeig4+okNTabZWKydWCjOcPxUVPe6222OYsHg7dMTuhqOecMK5un+oGJTM3ANEBZaBa5YvqCorW0/9ntHZWFw6vdX/MMD/nvqKoucUyC1CqKiFNFBaPi85D1l91c4c7zbvpAvQ2ol615ivGZSUdZbq6c8XMggIixoN6c7vgvqDmdqFRd+VQcOV8S6GQIx+0FGZIe1+y8EiEUuyFlj2MDGKRkZLDTkEFJ+gaQBGF0TdtHeqPsUE12479NPXDLeOZp184a7xw/sLKBfg9f/5CzcYSsUImktuo3P4DqRu6uYNprZ4WsovLqHLdZpHXUcsgJZdX/aFt0y2w0mKOnzSfPPCxeMZF5RJpSrXJG61BkzsHvJfRT1lMEMe8PfU8C71LQ4QxIGEqI4hriFF5VrvUJijNYj565oyxRoUUGqNkFLpOY1dhNkSJAprkAw3zuJHqtbypspni1jRKYaUOBSxRa/dcq9XSO/mDCOg/2zDwX4WcCK0fgfibRtbVSJ7147VxeVTP/Wl8UMIB2VnilGBHsTZAi5E9HV56Ks4RuRWWmXDOKs7plOB28jS1oixaFi2n4wU/QNHSA4UzPTiR2mfXSLEaLEpyAYAuxtqzlE0TZO6sKR+nxCoEGEGsItOswnwcwPmmn0ZjExMWEJDW7iqcLDaoG37XxnG3vc2mbVNCHE+3wZCCSA8n8ipk7GaJv2v0BYw2VotsCXLBh79gZlcUoQq2dcRJQYyAlBb9rx4AVrP5J8S6Zn+S67Yk/mLLhEyu3YKGoC3vXe2K5auKydklYAkAbhg7kK/9/2AK4sSZ8vg8prQrPZI5gtR1yRl78YecRwBjhyq9axLktmiq26AsHc61jSzMkj1RUEQE1sQu7cKxyF5SZsqSbb2HJ8wal9YLDC237bo9qJSEvavdXDzOwQ+25pj3mkNAYguSyTfhCotK1eAkK+6Mo2hqCWfSMCp7QJA2GqVM3feMo6nPHCodtmWFdsyGvsv7nlBlWz4Rjhd6EwwVyJiIjWwgWEFayWkZsoBcr4K4QoPaLhxgCRQtExd8hIZiRpbWMEP4ZwlAiSJCsVVCxsvMZvNnkR9a5MuwlsEzEr9hpKVLMYkNhD8pumzhxrO0gE9axAfRwK1sJJTOnVpyqVdY6tWTUvdBQbhcgHC53mZVmKFTTYNNik847US4mi428iJ9Iy+qUCmlqAI/4lAjhKomCaibVHYWQDcDADpHl/GSA09+mTN8mTNvmXNgmfATApaAyCtMsVVlqhicJIHBB9votEyB1Rh2V1hhA+aIFdOxNIC44iQrluvlkENBbDglmyIAGg+3hzm22DrB6Ui9KbaljSWTcvr02a6GBK+ZDoMk3wdbNfuYGP6bulUki2W+gFnBM244DFii7mIhvrukuIQNAEg+S5kFbmuaouRYNqOCYN+FdcSFZY7kMvHftqvRBdoxnDxVm2DEoLajF7eAhzR7O4mfMQdCDbzTbBiLDGBdBWObuXQyFOsI88Qt10nmFG+THysYMLvu2mRVC/U8ZfNwW6SE3lIyfKOwrL3GEOqTJ0ASrq6ypRIAX6KuhyelFzC49bKwt83infzDEiyVOn/shvfR/K3bTvWt26hJR8NJrfQS0ZVeFFKMHNLJWLdXNeaKJHYMo8U3A/VP9aqHpK6PwiHblVJT6wjABpRuoSSyfEKJmfPxEFsVceEyvMUyZ4R3P9Ek8UDKfVAqthsHvudneYove3rlm2I5oItvDgrcspkCAn4UNvHFv6nu4WBmk+36YFv12xQBFB+04vQmljlSOi+6snEytptZLPQisHZbffFE07apxmMCFaC7nVZRsTstGcExLCqyocPRK1KMmSdJ3tCHIa14jf1UDhVD5TrDhT3IxcZP7/qAwD9Dm4FsQmZ76hRfGeFHsZSXsTejbLuZVzlomvYKxh3z4gN+4A9Rxky+O22CdqUo/JbZHHtubIrboZG5j9MOeKeoWIxJpAh8rseCdNygcEg0XV9HePB1VP2rUK9BGlSlmsqp5SVEX5SMqZQ0pIdnMLUsvdSpKOZPejjNbCtyzJpXbFWYG93eoPpSVM6az84TYOeNjd6p08UdBD+kDSxwd0So76NU5yDsXqs7FEdmL1UfWOUMTuk1oAayuRVEm5b5mFgsQwpe48KjzRIxW1gv+kwmTAAV5TXeVHo4UNrJYS2S9efEdgI2ypkFenMxROduwh8enAlIRZUUWHC1HPqpF01ZotQyiSJpePAjRf5RBvqfRsG0dOEoWBMTmTE5PporGFR6KJlbyKafOoSyTbodN9PJaOQrsaPnoZy7NQD4CnBPkAPhjRZhiZ30AQsiObJimnpUyAI3TvEwmReB9MlAAEhP8R6Crk2G/nQcDfFJAfoPlBo52wbztibzZohhJhlaFRhN82UIpGGslZJrc5/26LVWhwe9fbFUtHA9fTTVq6zz0oXMe5kVuJt4b8vLe/yRH39CSY3ymy0xmz7rS6hDW4cBykLAZbBp4WsiagBrhQHj7w/OuaBWSxodnhsE+QszWCt20N8iRngfj19Ea44jUJQo9D29fgusSLcJOq00njRaJAAIDR/bUa8eGpSwAmG1cgq0awCCgg21Be/YOZVPx/dovdIzdJ0dMHs+NyTcvMzHQOnoqREIhQXZOOfJCpHwGFWW6K4EydQe9PGXQyibI9Mw9vNTPvjh2/f2BdDe6WZ7dPBTw9pHkL3Gwco+AYJPWLI8vPWfh++8dv/eV8dX3j189ytjXyqHojeXAvsAFhx9+fnxZx/Mna0L4AEmVMEk3e5jyVuVKVygUTp3/s47SvUWllhxU+07GlIwa6Uh9RI/zsjM9+lrh5ZDdsVx7IoNoAUultHldyGbTyVbEPGF2XlsJdaQcXgQLfXN49c+mb3++8Nrfzi8fn12442jux9zZs0+/ObovZfpuxc/fPvG7NbvZzd4yzh89+fH3709e+3u7PobRzc/M54738UZ33x1eO3t2ZtfHH9/c3b39uz665yB/3PlJWEe3biJl02uQMYyc0+DCddezPrn6TY0hClp3/wbE193BHHfnF3//MGVa7jpKx8p7H749ub9O2/OPvjD7OpvHlz5xexPv+BEHL/8Hpx4iZQFWEAIRQEdIUAvKfimx7dfPbr5zuzOx0dvvQZ7Pbj36+PPPgQmKJiGHnEayCwAhJy4cRtQgoBw8aYYMZmK2GVF7OEbr80+e08Pbmc33ppd/2T24ctHN149/v439+/e5DgshC/Coh+7gwGhhHH07Tuz619r2wAglCvySinD/dL80RK2mjlQzatz91oaL3la7TaOfCt4TpgEWgS7MbpfM0sCgKf/6a9BMjX3q0Uz5TAcoTY4EiIKp88yFK8JjSRNKsTWF2h4ySB8JY/BjQe3rhzf/nmF0xqqgiu5atcQUR+0LOlWkQs96OHsT79Cm4gzDi6Gh9e+m139wtinVIwvBtt2/84nQqNvgtYaRze/PHzro/vf/xb5ePULFO67Nzh/QclBlJHFV2/O7t3l8FESivaO0J14HqMvT4Pdd/1gkvADuORDjDosvB2XqWjilfJQjmExD82t8iIniaa0lB7Ek81AOk69W3DsYrixT4iAnyiwZ2Acfvo7tHSkB8hMQPSghmJeNNqrPNyPJ5n+LARWNzHudzDLwfdG2IGh7YGzzwNBiMvjrqk9HAHe0IB8QNKEJHwMOnZQTjPw+Q2WJmRMzeuSE4oLBSbNKd5Nw1hNviEPp/ZrN5xR4G+Pv/7v4+/B+L03+/q/Dm9dA5s6u/GmsPcgGtxCgizNbr8E5gNm3L9zBUTs6Oadw1t3uRzBzOPPv569cvXwd1cO3/+oXoJOfAkxoiNWJQRgI/d1Up/jtPYbe218mYRrz4hif69yaW9yAz/7/pUHH9yDszfaf77yNs017n/zbw/e/bIU26u3P/rjIUBIq7zJoBYM1uf3ZtffERp3dfbpu8evv/TDt9fM+QFtcTNNLEpHK8SiN+fsMFjisnz4qy9Blo19FCbxsOPAOP741Qf/8e9A+Z+v3D7hOIhe1D/9aUhVnNRrhvonHJUFf2U8z2+Zkz3j3LP/8PQFIwqDvQY9e9hmMtsDo8joueFfw3EaIdtRPT6kwBWgo2GDFxb6KsBqji+hwsVWnDBQq/6jzUeXUW2wc/lR51GpbzAAEpX5bgBdsEk/l7Eq8lHa9IIoBZUe1jCM4df+eJCGyNTPyL8QhS06Xrs6M6/78lmqXJc/0GsUngnyKmbtlxuleJSFdGQe/fL9w6s3iiKC4nrn479EVsWNmHrqUZFjSSedMbKgV8fYwA8vaTxBSXkqi8a+x626R/93H1SZRVD8K9NYHyA7iB/IFVdlQ/kEyXj6dtB87QGLx4NcdAMw92CBnuQOsGJOlSlVSQh3LsDw2evvz7eGbNdjcWY8TX+QYjfFvlKRVjrbOdsSHR/+8fjLj5AIemYKMOymQw7JcQ56kN3seotI07yK4mAPFWgS0kmJuosTXSotJjsQlFxkLgdOLgX474ngHiJb1ume7+Cv3OP5icFnrOjjmIb2eCp5Qtb40AnjolxRc1QCg54Ql8Nbn4j4jSdcJU/6y+8gDNbTHMg9CplOdZ889uHyd3Tv7cPfvo+e/ZU/1gSJsBeXUEi5pXDLAS5Rxr6UPtnPgwVYwEML0X0TOV5iTPkIi1yqJwBL2GLDXqVKiN9aW/Kx6MIFm+pqjoNpuOOY8r0J5uRL/wv3XdYX', 'b86c0096eb15ee63e39ee5485b2872a167b6639204b554dc76016d97832159e5'),
    'extractor.py': ('eNqNV11v1EYUfd9fMU0fbNPFbCpQ1cA+IARvVJFA7UMUWRPvOJlm7LE843w0WglVFIqAkopCUaFCrarSPiCqFpW0QeK/tNlN8sRf6L0ztrMfNsVSsuuZO3fO3Hvm3LszMzOf8KQn14/Df5Yy+JdowjZ0RkPNZUIUy9Z4yE6TjOWKEb3CiMrTVHDWIxfnT5L1jGuW+TMzM60okzEJgijXecaCgPA4lZkmNEmkpuhMtVrFmFTlN83iNOKCVe88ZtZTSvWK4Eulm3l4rdbHIU0DLYM4PUmoIqHMWKvV6rGohB6gT+UqmWchU20Spar7kUxYmwAYHm2aF2+uReDhEU4TrnCO4ARg7hEX37jiidI0CZkLNm3CE+0RmRnLWXKma1bCx4cdeAp/+GSUQ7Q+piJn57NMZq4z2Hmy/9X1wasvDn/YHT54Rmb/uXLXrCJ7f908/Pb5/ndXh/eeD+/95njGSwGddIngSrs9Hmof47LKNpWLwXBTz8+YkmKNuR6JAFQK8Mp1nvXCYq7BhT00AncFjZd6lLA1SPScOa21BFe50LjfwqIFoGmmIctdkxM/luBEJjx0rT1uiJzZaBdb4uYsyWOWUc3KyI+EBDMNEUzzJTjQinG8sNgudzuCAONbjl3uzAGKrPDltYkDmHSuYNiJKCS458CYzHWaaxxcWOxXvvDg7pazChCNOVijsUEMAwXyEYdpJgEvpHsZ7er273uVd0GVDmDFMkDGkHVa1RSSsJxxwf0ycG6JKvi/zvjyih6JSGmepz0MWc/wU8NlERNG+CQyETKkYnzvGrP1poyNPsB53I6csfsZwuPS4xNHO0M6/uypaTQ2XXDRk6mpydiA2ymbqJSXLol54s62SUw33E7bgjpRBMFDmBYfE3CdOlN+JrJc7urYzC7bBJoUTGS+9kAEPDDIdKLBzjWG5D2TPPioIB8rEwk4BUsqpsPEbKfT6JkJmiqGMG2ci+vVH8+NjWnBiCNS6WxzPAcQGJQgu7fPldE710hTMabyKOIbvpDrLIOJd7rE8VE2nelkTovVwbMXh1duDG/+Onj6YPDoF9AmYhaT4f3re7t/OuOgixuIF1fLlIdzJYZ1rleChMYgns5WiQt0oB9sWXh9HxTc8RpiZjTGeASNMfaoMSj2/rmzFy/5UHli5Xr9ycjQZNNNfbYBuqlGpbHA6a/hSWGm5pZZCfKLG2nVoeuoVZ6mRm1iYBfwqesMXvw+fHRjsH1rsH0bC+Hrl7dg6ODFHwevrr9++XBv5/bBz9f2H963Afv3yucTp0RCT28PFMupCLCodO1Bl5nG14JlfkozoKepZ7WXuqpLI57K8tRwi6eSr1AGZeJ/qoDteztPIfujxQv4MFW8alJoI7ngAAJnkYwCmjLFir1anTiUQjCo4FSI4tTtI0EFZu4O7twnF8+dnYd0gGJ0/A9OedPbY9JDisWoTdyCQzb/3nihKklRkMmrj5KBiMJqoC4Yh4u1lorGqWDVaSKeUME/Y65Z2bZ0rid81IPuBCDjNkVX5MereF9SN80Y0L/r+A5IkYVsbhK8OYHTJDrNj71O4A8IpTkVEMsez7pjLKtHKZUfCqngSvfqDRAvHMG0KHicZivl0xRbTtfFtyo/9QtMNE23iV1fsaIIdtvmxedJJMvvcAYaw8wR794qShXTImf/m8fDL7fJViU4Rdr7jmUdxN4yDJQfquT7p9r2o+EAPSZKvLVsHY0BUtREqJ6NcNEhDZYroOtYrxrP9i6xDb6CG2lWAJOgk1fARPiaChpiu0uMWkLzQ6AsgeacfoO/PBF8lZEVmvWgN03g5kKPSYWSZF1m8CbR24Wzl0/AH1mTAm6a8hv9wUEssHES1NrXi+aIJ4TzFn6qHrQkYJNxQznIw9B2GYV4dBewTUxHSk21w+L/sa6sKJEpKXe2Qd0Gd27tP9wZPvr74Mm1wx+/xtJi9JdsHZG5Ty7MXxovK2wjZKkm580HdirwywjGxgOGCI84VsGcjmohMphsiGnMTWscyNXu5Sxnk02LlfriKEbuoeDrzZS5AMDzA9MEBEF/jmzBQN+plht5FJvTGG0SgzdcBBz+X3jFr5oyz/bVa/qVYKdrficcO2an+o1LR1vPmh8ZphV1bDMw+Onq/va1/d27w+8fT0l3XQc629BqjjaVkx3/VIdZdJZFQFr/AYXN1c0=', '6565a4fc10a165302bf4c4b2fd89c1ef9525638208339d67fc71de34af52fc5a'),
    'ros2_video.py': ('eNqtVt9v2zYQfvdfceOATUIVxUnTonOrAUPWYXnoGjRFX7JAYCXK5iyRAknV8YLsb98dKcly4qHZML1IOh4/3o/v7sgY+/D+6hSi858/xHCum9YIa0V50fClgF/T05cv4OhHeHd5loARnZVqCU3B29zpvGnPvre0BBsjnTApY2xWGd1Anled64zIc5BNq40DrpR23Emt7GzWy7QdvqwzXeGGPyeatpK1GP9lIwJuy92qlp8H0Ev8HdEmVgG3UGgjZjPyLf/4/vLiHDJgxyvBy+OCN8LwY0ke5sXoMZvNZqWo8Axjp/LcK0aNXcaLGeCDXlqhrDY5yuzxg5glgJGEWjpXiyOhSskVBTCqtGm4S6Dkjsc+UoQlK0CQ65Mb+CaDk4BPj+HSCvjE6068NUabiBHqZ7kcIJ0s+RpKWXbrTi1Z7HdqdPIMnsEr+PrzLQiFEbNd7bMCFBphcLN1vGk9nEK4kJm0Uy0v1jklIWJvLlhCZieg4+v5zWs891k4WE3xK4OBzmU5mhahHjyP4Tv46/l/OcDvqRqHuyhoekF46iY1iCHb6DP7fc7itBSFLkXEOlcdvUIcZkRb80LgUq03wkRxj6f+X8OMQMYrsi/ZM6+nVSmcKFxe863uXERE3tHpyunWEmmtxkzg260EBE2QSGVhHJfqNdiCKwuoolW9BWQUdGqt9AZPxXKxI6sQThaWfBAuCtQgZd1SDRcgla+OlGo2R4E2pZ1a1BNTt0TK+e38bCemp9DKSdWJUVjIMvFx9KjdycsIQROYx6NGfmhd79a9wQnkg06DwT+sZFNelpH/jKfGeglkGewqft/qPj3MaHvKpjt5XUeOYtJHjSLlxhid//TuKj6MJJZ6LGKutk8HmQJMC6YWzmdeG7mUitcgbp3hhUMs5DD1uM2Ke0Y00lInnh1uFWTDGr7IUuixT6wFAsrXENa2HNs4L/kCGHKUiiT9Q0sVYVNzog+vjeO4526h65rIS7HzREmgNXpJXS/7TSvR+7bhimqTJIGGaPx6SOlH+vk3ZJycMXw84md2kJ9fpeNTKPlkWj6VgJMIFX1TpEfUU1deIJPKoIZpxpHpo+mFD73B/kObSHf/IN+BaMzgSf8wzBDi+vR0cRM/KGw8Acsrb/m21hx5QDkLWEmA3TkXD9QfkrMzYpD4NKY4T7AJ4Su38k8cjweE8bSB+kN74vUFEIhndWcKBKjaPdYFMfpKt4FeKU7xfF1/EdEAjXOOIn/HggJbUH8ftJH/ZFBnUcwqjr20pJrA9tt2joTXN/gbGjKp+B5yH8aH16FW66HSjXSrXOHgiyp218ssXmju83DtyOcpXlD6eU0Xnf2C2e4z3IOn4lZaZ6NHbYh8SrsW0yOiYH/G7Fq2rTe/wQxgrjNGFzTblXxFFY/ZRPc2OPNTFh9qawF2XMHod7zOMeZDESyFo98+dikyTCjns7LXkIm7J/AmmyLg3w9zfB448qiBke6Km87CyZHfMDE12HdNOgwLYAK/GxRj35m0rT7Vo5LFWw7OzMGrihouUnHg/EOi+8LCvkKpxT3DFTVt1pTdNsL6quRtxlJqqH3eaIXaa47ZsF0V1jFeTvKaUSJMthfE3UnapkWtLZKonMy/wBZPczJjt+I98Hdwuv1GpJgMDiYhHKlUlR6+8Tje4MoudntHoy1E4AATfIk9HW0a3MertHIMWxfmDYW1VOt97dlXSNoVBbKTDfo2u6Zq7Ddjrfmro81qgSMpuBEns8P32IHmFfvkB97al5nPH0ZYYHoMx3zeTbHuwwEo3UXgHn65vBpqQtwWonXw1r9kuJOhbPGIhf3pnolY8G7biggV4zT3TSDP7xdwh4L7cFPwLKv3i9ynddLs94uDVvEO6kPcT/1cr7OPptvvmn3Z/g0NWCCl', '76647bfb1a8863c69e2204bdb334c34a7ce3319319d2c759b6ca6a8f31695754'),
    'worker.py': ('eNp1Uk1rg0AQve+vmNvugkiSY8Bjeyy9i8jWjOkWdWVnDAkh/72zapq00AVR38y+9+bD92OIDHShDL4oDKqNoYfR8WfnP8Av0Xf5VZKSJzz3A2Fks8mAOJoUM3Xd+g7r2uYRKXQnNFZyIw5s7f+M63cMtKtP/oBhScUzR9dwiPfkFZg1SCmKDRQzg0meXDyeym0lOiMJ/hYGBN/CT2hXQVGAdhMHDdiRRAc2T2GrOF72CuR07hImFpaHp/yAjCK+hIyI2zlVJO7Zwo7HoBeKdKQJwvHLtinlZpWBmCzkseWmek4v9UKmxezCNkeT3b+0T9buCglKzh70Cs8Njgwv88uHARzBSrXQXDWFKTao9/MYU10ZaGLHEwmmWye2D1owsTVOnMBSCtA9ErljutfqK19GNGjzuh5cLwtw28MVb/qmxpi6nDYqP0z9SEZUM8CBpoi1o8b74tVJdbIfaRR49rJRqalzM1Yby+RoahrRXIe3teobm//lpw==', 'f7123a096a49d2fd8309e94ad8a2d7e7d51fe37bc1a348cf6a761b481cae3305'),
}
for name, (blob, sha) in _FILES.items():
    data = zlib.decompress(base64.b64decode(blob))
    assert hashlib.sha256(data).hexdigest() == sha, f'{name} rusak'
    Path('/content', name).write_bytes(data)
print('Drive terhubung, skrip siap.')

Mounted at /content/drive
Drive terhubung, skrip siap.


## Langkah 2 — Atur folder
Ubah path sesuai Drive kamu. `FPS = None` artinya otomatis (baca `session.json` di folder tiap file, kalau tidak ada pakai 30).

`RECURSIVE = True` membaca semua subfolder. Hasilnya meniru struktur subfolder sumber di `DST_DIR`, jadi nama file yang sama di subfolder berbeda tidak bentrok.

In [ ]:
SRC_DIR = '/content/drive/MyDrive/mcap'    # folder berisi file .mcap
DST_DIR = '/content/drive/MyDrive/mcap_output'   # folder hasil MP4 (dibuat otomatis)
FPS = None                                       # None = otomatis, atau isi angka bulat, mis. 30
RECURSIVE = True                                 # True = ikut baca semua subfolder di dalam SRC_DIR

## Langkah 3 — Cek daftar file (belum memproses apa pun)

In [ ]:
import os, sys, shutil, psutil

src, dst = Path(SRC_DIR), Path(DST_DIR)
assert src.is_dir(), f'Folder sumber tidak ada: {src}'
dst.mkdir(parents=True, exist_ok=True)

it = src.rglob('*') if RECURSIVE else src.iterdir()
files = sorted(p for p in it if p.is_file() and p.suffix.lower() == '.mcap' and not p.name.startswith('.'))

def out_dir_of(p):
    return dst / p.parent.relative_to(src)

def is_done(p):
    # camera_0 selalu dipublikasikan paling akhir, jadi adanya file ini berarti semua hasil file tsb sudah lengkap
    return (out_dir_of(p) / f'{p.stem}_camera_0.mp4').exists()

todo = []
for p in files:
    done = is_done(p)
    print(f"{'SKIP (sudah ada)' if done else 'PROSES':17} {p.relative_to(src)}  {p.stat().st_size/1e6:,.0f} MB")
    if not done:
        todo.append(p)

print(f"\nTotal {len(files)} file | akan diproses {len(todo)}")
n_sess = sum(1 for d in {p.parent for p in files} if (d / 'session.json').exists())
print(f"Folder dengan session.json: {n_sess} dari {len({p.parent for p in files})} (sisanya pakai FPS di atas / 30)")
print(f"RAM bebas: {psutil.virtual_memory().available/1e9:.1f} GB | disk lokal bebas: {shutil.disk_usage('/content').free/1e9:.0f} GB")

PROSES            ego_gopro_21912_20260930002459_0192_806e67_RAW.mcap  660 MB
PROSES            zyw/ego_0000.mcap  13 MB

Total 2 file | akan diproses 2
Folder dengan session.json: 1 dari 2 (sisanya pakai FPS di atas / 30)
RAM bebas: 12.4 GB | disk lokal bebas: 94 GB


## Langkah 4 — Proses satu per satu
Aman dijalankan ulang: file yang hasilnya sudah ada di Drive dilewati. Kalau ada yang gagal, file lain tetap lanjut.

In [ ]:
import subprocess, time

WORK = Path('/content/work')
ok, failed = [], []
t0 = time.time()

for i, p in enumerate(todo, 1):
    rel = p.relative_to(src)
    print(f"[{i}/{len(todo)}] {rel}", flush=True)
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir(parents=True)
    try:
        local = WORK / p.name
        shutil.copyfile(p, local)
        if (p.parent / 'session.json').exists():
            shutil.copyfile(p.parent / 'session.json', WORK / 'session.json')

        r = subprocess.run([sys.executable, '/content/worker.py', str(local), 'auto' if FPS is None else str(FPS)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError((r.stdout.strip().splitlines() or [''])[-1] or r.stderr.strip()[-400:])

        out_dir = out_dir_of(p)
        out_dir.mkdir(parents=True, exist_ok=True)
        outs = sorted(WORK.glob(f'{p.stem}_camera_*.mp4'), reverse=True)   # camera_0 terakhir
        if not outs:
            raise RuntimeError('tidak ada MP4 yang dihasilkan')
        for out_local in outs:
            tmp = out_dir / ('.' + out_local.name + '.partial')
            shutil.copyfile(out_local, tmp)
            os.replace(tmp, out_dir / out_local.name)
        ok.append(str(rel))
        print(f"   selesai ({time.time()-t0:.0f} dtk total)", flush=True)
    except Exception as e:
        failed.append((str(rel), str(e)))
        print(f"   GAGAL: {e}", flush=True)
    finally:
        shutil.rmtree(WORK, ignore_errors=True)

print(f"\nBerhasil {len(ok)} | Gagal {len(failed)}")
for n, e in failed:
    print(' -', n, '->', e)

[1/2] ego_gopro_21912_20260930002459_0192_806e67_RAW.mcap
   selesai (304 dtk total)
[2/2] zyw/ego_0000.mcap
   selesai (311 dtk total)

Berhasil 2 | Gagal 0


## Opsional — Diagnosa satu file
Menampilkan daftar topik, jumlah pesan, dan format video di dalam satu MCAP. Dipakai kalau sebuah file gagal diproses. Tidak mengubah apa pun.

In [ ]:
# Isi dengan path satu file .mcap yang bermasalah
FILE = '/content/drive/MyDrive/mcap_input/NAMA_FILE.mcap'

import sys, struct, collections
from pathlib import Path
sys.path.insert(0, '/content')
import mcap_to_mp4 as core

FILE = Path(FILE)
schemas, channels = {}, {}
count = collections.Counter(); first = {}; fmts = collections.defaultdict(collections.Counter)
for op, rec in core.iter_records(FILE):
    if op == 0x03:
        sid, o = core.u16(rec, 0); name, o = core.mstr(rec, o); enc, o = core.mstr(rec, o)
        schemas[sid] = (name, enc)
    elif op == 0x04:
        cid, o = core.u16(rec, 0); sid, o = core.u16(rec, o)
        topic, o = core.mstr(rec, o); menc, o = core.mstr(rec, o)
        channels[cid] = (topic, sid, menc)
    elif op == 0x05:
        cid, _ = core.u16(rec, 0)
        count[cid] += 1
        data = rec[22:]
        if cid not in first:
            first[cid] = data[:24].hex()
        if count[cid] <= 50 and channels.get(cid, ('',))[0].startswith('/ego/camera'):
            try:
                payload, fmt = core.compressed_video_payload(data)
                fmts[cid][(fmt, payload is not None)] += 1
            except Exception as e:
                fmts[cid][('ERR ' + type(e).__name__, False)] += 1

print(f'{FILE.name}: {sum(count.values()):,} pesan, {len(channels)} channel\n')
for cid, (topic, sid, menc) in sorted(channels.items()):
    sch = schemas.get(sid, ('-', '-'))
    print(f'{topic:35} pesan={count[cid]:>8,}  skema={sch[0]} ({sch[1]})  encoding={menc}')
    if cid in fmts: print(f'{"":35} format video (50 pertama): {dict(fmts[cid])}')
    print(f'{"":35} awal data: {first.get(cid, "-")}')

## Opsional — Diagnosa gambar ROS2 (file GoPro)
Untuk MCAP bertopik `/head/camera/image_compressed`. Menampilkan format gambar (JPEG/PNG/H.264/H.265), jumlah frame, perkiraan fps, dan ukuran data. Tidak mengubah apa pun.

In [ ]:
# Diagnosa lanjutan: isi topik gambar ROS2 (sensor_msgs/CompressedImage)
import sys, struct, collections
from pathlib import Path
sys.path.insert(0, '/content')
import mcap_to_mp4 as core

# Isi dengan path file GoPro (.mcap) di Drive
FILE = Path('/content/drive/MyDrive/mcap_input/NAMA_FILE.mcap')
TOPIC = '/head/camera/image_compressed'

def parse(msg):                       # CDR little-endian
    o = 4
    sec, nsec = struct.unpack_from('<II', msg, o); o += 8
    n = struct.unpack_from('<I', msg, o)[0]; o += 4 + n
    o = (o + 3) & ~3
    n = struct.unpack_from('<I', msg, o)[0]; o += 4
    fmt = msg[o:o + n].rstrip(b'\0').decode('utf-8', 'replace'); o += n
    o = (o + 3) & ~3
    n = struct.unpack_from('<I', msg, o)[0]; o += 4
    return sec + nsec / 1e9, fmt, msg[o:o + n]

def signature(d):
    if d[:2] == b'\xff\xd8': return 'JPEG'
    if d[:4] == b'\x89PNG': return 'PNG'
    if d[:4] == b'\x00\x00\x00\x01' or d[:3] == b'\x00\x00\x01':
        nal = d[4] if d[:4] == b'\x00\x00\x00\x01' else d[3]
        return f'AnnexB(h264 nal={nal & 0x1f} / h265 nal={(nal >> 1) & 0x3f})'
    return 'lain: ' + d[:8].hex()

channels, want = {}, None
n = 0; fmts = collections.Counter(); sigs = collections.Counter()
sizes = []; stamps = []; samples = []
for op, rec in core.iter_records(FILE):
    if op == 0x04:
        cid, o = core.u16(rec, 0); _, o = core.u16(rec, o); topic, o = core.mstr(rec, o)
        channels[cid] = topic
        if topic == TOPIC: want = cid
    elif op == 0x05 and want is not None and core.u16(rec, 0)[0] == want:
        t, fmt, d = parse(rec[22:])
        n += 1; fmts[fmt] += 1; sigs[signature(d)] += 1
        sizes.append(len(d)); stamps.append(t)
        if len(samples) < 3: samples.append((t, fmt, len(d), d[:16].hex()))

print(f'{TOPIC}: {n:,} frame')
print('format :', dict(fmts))
print('isi    :', dict(sigs))
print('contoh :', samples)
if n > 1:
    dur = stamps[-1] - stamps[0]
    print(f'durasi : {dur:.1f} dtk  -> kira-kira {(n-1)/dur:.2f} fps' if dur > 0 else 'durasi tidak valid')
    print(f'ukuran per frame: min {min(sizes):,}  rata-rata {sum(sizes)//n:,}  maks {max(sizes):,} byte')
    print(f'total data gambar: {sum(sizes)/1e9:.2f} GB')
